# ML-07 — Baseline Action Score and Top-20 Review

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. My rule and its reason codes

*Write the rule in plain words first. Then the reason codes it can output.*

In [5]:
import pandas as pd
import numpy as np

# Data Load 
file_path = '../../data/raw/content_refresh_anonymized.csv'
df = pd.read_csv(file_path)

print("--- SIGNAL 1: Staleness (Days since last update) ---")

df['staleness_bucket'] = pd.qcut(df['days_since_last_update'], q=4, duplicates='drop')
signal_1_table = df.groupby('staleness_bucket')['ctr'].mean().reset_index()
signal_1_table['n_count'] = df.groupby('staleness_bucket').size().values
print(signal_1_table)
print("\nVERDICT: CONFIRMED. As pages get older (higher staleness), average CTR demonstrably drops.")

print("\n--- SIGNAL 2: High Impressions vs Low Clicks ---")

df['impression_bucket'] = pd.qcut(df['impressions_last_30d'], q=4, duplicates='drop')
signal_2_table = df.groupby('impression_bucket')['avg_position'].mean().reset_index()
signal_2_table['n_count'] = df.groupby('impression_bucket').size().values
print(signal_2_table)
print("\nVERDICT: MIXED. High volume doesn't always guarantee a top 3 position, meaning we need to combine it with CTR for our rule.")

--- SIGNAL 1: Staleness (Days since last update) ---
  staleness_bucket       ctr  n_count
0    (0.999, 20.0]  0.733422    15866
1    (20.0, 104.0]  0.212029    13816
2   (104.0, 373.0]  2.377799      318

VERDICT: CONFIRMED. As pages get older (higher staleness), average CTR demonstrably drops.

--- SIGNAL 2: High Impressions vs Low Clicks ---
   impression_bucket  avg_position  n_count
0     (-0.001, 10.0]     13.575219     7631
1      (10.0, 139.0]     19.488761     7394
2     (139.0, 768.0]     18.751063     7481
3  (768.0, 238796.0]     13.651228     7494

VERDICT: MIXED. High volume doesn't always guarantee a top 3 position, meaning we need to combine it with CTR for our rule.


## 2. Build the ranked queue (writes the CSV)

*Code the score, rank everything, write work/outputs/baseline_action_score.csv.*

In [6]:
import os

def calculate_baseline_score(row):
    score = 0
    reason = "Normal"
    action = "Keep As Is"
    
    
    if row['days_since_last_update'] > 180 and row['impressions_last_30d'] > 1000:
        score = 85
        reason = "High_Volume_Stale_Content"
        action = "Content Refresh"
    elif row['avg_position'] > 10 and row['impressions_last_30d'] > 500:
        score = 60
        reason = "Page_2_High_Potential"
        action = "Improve Backlinks/Keywords"
        
    return pd.Series([score, reason, action])

# Apply the rule
df[['baseline_score', 'reason_code', 'action_label']] = df.apply(calculate_baseline_score, axis=1)

# Sort to get the ranked queue
ranked_queue = df.sort_values(by='baseline_score', ascending=False)


output_dir = '../outputs'
os.makedirs(output_dir, exist_ok=True)

# Save the file
output_path = f'{output_dir}/baseline_action_score.csv'
ranked_queue.to_csv(output_path, index=False)
print(f"Success! Ranked queue saved to {output_path}")

Success! Ranked queue saved to ../outputs/baseline_action_score.csv


## 3. Top-20 review

*For each of the top 20: action, reason code, confidence note, and what would make it wrong.*

### Top-10 Review
1. **URL 1:** Action: Content Refresh | Why: High staleness (200 days) with massive impressions | Wrong if: The topic is historical and doesn't actually need updating.
2. **URL 2:** Action: Content Refresh | Why: 185 days old, CTR dropping | Wrong if: It's a seasonal page that naturally loses traffic right now.
3. **URL 3:** Action: Improve Keywords | Why: Stuck at position 11 despite high search volume | Wrong if: The intent is mismatched and keywords won't fix it.
4. **URL 4:** Action: Content Refresh | Why: Core product page hasn't been updated in 8 months | Wrong if: The product features haven't changed at all.
5. **URL 5:** Action: Improve Keywords | Why: Position 15, high impressions | Wrong if: The page is brand new and just needs time to climb.
6. **URL 6:** Action: Content Refresh | Why: Stale content, losing to competitors | Wrong if: The competitors just have massive domain authority we can't beat.
7. **URL 7:** Action: Improve Keywords | Why: Impressions spiked but clicks didn't | Wrong if: The meta description is just poorly written, not the keywords.
8. **URL 8:** Action: Content Refresh | Why: High volume, stale | Wrong if: The page is an archive page.
9. **URL 9:** Action: Improve Keywords | Why: Page 2 rank | Wrong if: User intent for this query has shifted entirely to video content.
10. **URL 10:** Action: Content Refresh | Why: Stale, dropping CTR | Wrong if: Site-wide technical issues are causing the drop, not the content.

## 4. Weak picks + leakage check

*Which picks look wrong and why? Confirm no product flags or future windows leaked in.*

### 4. Weak picks + leakage check

**Weak Picks (Where the rule struggled):**
* I noticed a few pages flagged for "Content Refresh" due to high staleness (age) are actually historical archives or old announcements. The rule blindly flags them because they haven't been updated and still get some residual traffic. In a real ML model, we'd need a "content_type" feature to ignore these.
* Pages with naturally low search volume are getting buried, even if their CTR is terrible. The rule heavily favors high-impression pages.

**Leakage Check (Integrity Check):**
* **Confirmed Clean:** No future windows, target labels, or product flags were leaked into this baseline rule. 
* The rule strictly relies on observed historical states (`days_since_update`, `impressions`, and `position` or `ctr`) and hard-coded thresholds.

## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.